# Reviewed remediation
Start with the separate [read-only Preview Quickstart](docs/QUICKSTART.md). This notebook plans, rehearses and can change data. Use a disposable PostgreSQL copy first; see [operator safeguards](docs/SAFETY.md).

**Do not use Run All while connected. Defaults are disconnected and commit is off.**

In [ ]:
from remediation.connection_ui import ask_to_connect
from pathlib import Path
from datetime import date
from getpass import getpass
from remediation.runner import Config, connect, preview, export_review_tables, build_plan, save_plan, apply_plan, rollback_run, discover_event_scope
from remediation.reporting import apply_and_report, generate_committed_report, check_report_destination
from uuid import uuid4
from remediation.operator import review_summary, rehearsal_receipt, require_rehearsal
RUN_DATABASE = False
COMMIT_CHANGES = False
conn = None
plan = None
reviewed_rehearsal = None
print("No database connection has been made.")
from remediation.reporting import generate_audit_only_report

## 1. Choose the database and scope
Use approved schemas, matching CDM version, read-only credentials and private output outside the project. Keep TLS verification on. Confirm discovered scope; a site label is not a row filter.

In [ ]:
cfg = Config(
    cdm_schema="omopcdm",
    vocabulary_schema="vocabulary",
    tables=("condition_occurrence",),  # Bootstrap only; discovery below selects all visible event tables.
    cdm_version="5.4",  # Set 5.4 or 5.5 to match CDM_SOURCE.cdm_version.
    as_of=date.today().isoformat(),
)
private_output = Path.home() / "private_remediation_results"  # Change to site-approved storage.
if conn is not None:
    conn.close()
    conn = None
RUN_DATABASE = ask_to_connect()
if RUN_DATABASE:
    conn = connect(host=input("Database host: "), port=int(input("Port [5432]: ") or "5432"),
                   dbname=input("Database name: "), user=input("Database user: "),
                   password=getpass("Password (not saved): "), sslmode="verify-full",
                   sslrootcert=input("CA certificate path (blank for system defaults): ") or None)
    cfg, coverage = discover_event_scope(conn, cfg)
    print("Selected available event tables:", cfg.tables)
    for entry in coverage:
        print(entry["table"], entry["column"], entry["state"], entry["policy"])
else:
    print("No connection requested.")

## 2. Preview with no writes
Review the proposed replacements and manual-review CSVs. Already-standard concepts appear only as field totals. `ELIGIBLE_FOR_REVIEW` is not approval or proof of equivalence.

In [ ]:
if RUN_DATABASE and conn is not None:
    private_output = check_report_destination(private_output)
    records = preview(conn, cfg)
    for category, path in export_review_tables(records, private_output).items():
        print(f"{category}: {path}")
    from collections import Counter
    print(Counter(r["status"] for r in records))
    print("Private aggregate preview saved.")

## 3. Freeze the exact changes
Confirm scope, vocabulary, counts and meaning. The private plan holds row IDs; do not edit it. Narrow scope and rebuild if needed. Zero rows: read the audit-only PDF and skip rehearsal/apply.

In [ ]:
plan = None
reviewed_rehearsal = None
if RUN_DATABASE and conn is not None:
    candidate_plan = build_plan(conn, cfg)
    plan_path = private_output / f"plan_{candidate_plan['run_id']}.json"
    save_plan(candidate_plan, plan_path)
    plan = candidate_plan
    print("Rows:", len(plan["rows"]))
    review_summary(plan)
    print("Plan saved. No changes made.")
    if not plan["rows"]:
        print(generate_audit_only_report(plan, private_output / f"audit_only_{plan['run_id']}.pdf"))
        print("No apply: skip rehearsal/commit; review audit findings and exclusions.")

## 4. Rehearse and roll back
Review the exact plan above. With the authorized maintenance connection to the same database and unchanged scope, run this cell and type REHEARSE. No changes are committed.

In [ ]:
reviewed_rehearsal = None
if RUN_DATABASE and conn is not None and plan is not None and plan["rows"]:
    if input("Type REHEARSE to test this plan without committing: ").strip() == "REHEARSE":
        rehearsal = apply_plan(conn, cfg, plan, approval_hash=plan["plan_hash"])
        reviewed_rehearsal = rehearsal_receipt(plan, rehearsal)
        print(rehearsal)
else:
    print("Rehearsal not requested.")

## 5. Apply with separate confirmation
Only on the approved database with a restorable backup and no competing ETL. Run the cell, enter the backup reference, and type APPLY to authorize this exact rehearsed plan. Do not re-run Apply if the result is unknown or the PDF fails.

In [ ]:
COMMIT_CHANGES = False
if RUN_DATABASE and conn is not None and plan is not None and plan["rows"]:
    require_rehearsal(plan, cfg, reviewed_rehearsal)
    private_output = check_report_destination(private_output)
    backup_reference = input("Verified backup reference (blank cancels): ").strip()
    if backup_reference and input("Type APPLY to commit this exact plan with ETL paused: ").strip() == "APPLY":
        reviewed_rehearsal = None  # Single-use; reconcile the journal after any uncertain result.
        result = None
        print("Run ID for journal reconciliation:", plan["run_id"])
        result = apply_and_report(conn, cfg, plan,
                            private_output / f"committed_{plan['run_id']}.pdf",
                            include_evidence=True, commit=True, approval_hash=plan["plan_hash"],
                            backup_reference=backup_reference, maintenance_confirmed=True)
        print(result)
        if result.get("report_status") == "FAILED_AFTER_COMMIT":
            print("DATA ALREADY COMMITTED. Retry only report generation from run_id; do not re-apply.")
else:
    print("Commit disabled.")

### Regenerate the committed PDF only
If the database committed but the PDF was not saved, choose an approved private folder and enter the reported run ID. This does not update the database.

In [ ]:
report_run_id = ""  # Enter the committed run ID to regenerate the report.
if report_run_id:
    from uuid import UUID
    if not RUN_DATABASE or conn is None:
        raise RuntimeError("Use an approved connection with access to the original run journal.")
    report_run_id = str(UUID(report_run_id))
    folder = check_report_destination(private_output)
    recovered = folder / f"recovered_{report_run_id}_{uuid4().hex[:8]}.pdf"
    outcome = generate_committed_report(conn, cfg, report_run_id, recovered, include_evidence=True)
    print(outcome)


## Recover only the PDF if reporting failed
`FAILED_AFTER_COMMIT` means data are already committed. Never repeat apply. Use [journal-only recovery](docs/REPORTING.md) and a new private filename.

## 6. Verify and close
Inspect the after-preview and journal/report. Check affected downstream cohorts and derived data separately. Keep private outputs out of Git.

In [ ]:
if RUN_DATABASE and conn is not None:
    from remediation.reporting import primary_summary
    try:
        after = preview(conn, cfg)
        print("Post-run primary-field occurrences:", primary_summary(after))
        for category, path in export_review_tables(after, private_output, prefix="after_preview").items():
            print(f"{category}: {path}")
    finally:
        conn.close()
        conn = None
        RUN_DATABASE = False
        print("Connection closed.")


## Optional reversal of a confirmed run
Never globally swap new IDs back to old IDs. Follow the [journal-based reversal](docs/SAFETY.md) with backup and maintenance approval: rehearse first, then separately authorize commit. Later row edits may block reversal; restored IDs can be outdated.